In [1]:
! pip3 install --upgrade --quiet --user google-cloud-aiplatform==1.88.0

In [2]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

## Task 1. Set up vertexai

In [2]:
import requests
from vertexai.generative_models import (
    Content,
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Part,
    Tool,
)

In [3]:
import vertexai

PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

#print(PROJECT_ID)

vertexai.init(project=PROJECT_ID, location=LOCATION)

## Task 2. Define functions in Python

In [4]:
# Define the math functions with print statements
def add_numbers(a: float, b: float) -> float:
    print("Calling add function")
    return a + b

def multiply_numbers(a: float, b: float) -> float:
    print("Calling multiply function")
    return a * b

# Create FunctionDeclarations
add = FunctionDeclaration(
    name="add",
    description="Adds two numbers",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number"},
            "b": {"type": "number", "description": "The second number"},
        },
        "required": ["a", "b"],
    },
)

multiply = FunctionDeclaration(
    name="multiply",
    description="Multiplies two numbers",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number"},
            "b": {"type": "number", "description": "The second number"},
        },
        "required": ["a", "b"],
    },
)


In [5]:
math_tool = Tool(
    function_declarations=[add, multiply],
)

In [7]:
MODEL_NAME='gemini-2.0-flash-001'

In [27]:
sys_instructions=[
    "Fulfill the user's instructions, including telling jokes.",
    "If asked to add or multiply numbers, call the provided functions.",
    "You may call one function after the other if needed.",
    "Use add function in case subtraction is required.",
    "Repeat the result to the user."]

In [28]:

# Initialize the model
model = GenerativeModel(
    MODEL_NAME,
    tools=[math_tool],
    generation_config=GenerationConfig(temperature=0),
    system_instruction=sys_instructions
)

In [29]:
# Start a new chat
chat = model.start_chat()

In [30]:
# Define function to handle responses
def handle_response(response):
    # Check if there are any candidates
    if not response.candidates:
        print("No response candidates found")
        return
        
    # Get the first candidate
    candidate = response.candidates[0]
    #print(candidate)
    
    # Check for function calls
    if hasattr(candidate, 'function_calls') and candidate.function_calls:
        function_call = candidate.function_calls[0]
        
        if function_call.name == "add":
            a = function_call.args["a"]
            b = function_call.args["b"]
            result = add_numbers(a, b)
            # Send the result back as a dictionary
            response = chat.send_message(
                Content(
                    role="user",
                    parts=[
                        Part.from_function_response(
                            name=function_call.name,
                            response={"result": result}
                        )
                    ]
                )
            )
            handle_response(response)
            
        elif function_call.name == "multiply":
            a = function_call.args["a"]
            b = function_call.args["b"]
            result = multiply_numbers(a, b)
            # Send the result back as a dictionary
            response = chat.send_message(
                Content(
                    role="user",
                    parts=[
                        Part.from_function_response(
                            name=function_call.name,
                            response={"result": result}
                        )
                    ]
                )
            )
            handle_response(response)
            
    else:
        # Print regular text response
        print(candidate.content.parts[0].text)

## Task 3:Prompt the model to call your functions

In [22]:
# Test 1: Non-math question
response = chat.send_message("Tell me a joke?")
handle_response(response)

Why don't scientists trust atoms?

Because they make up everything!



In [23]:
# Test 2: Multiplication
response = chat.send_message("I have 7 pizzas each with 16 slices. How many slices do I have?")
handle_response(response)

Calling multiply function
You have 112 slices.



In [24]:
# Test 3: Addition
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. How many pizzas did they bring together?")
handle_response(response)

Calling add function
They brought 7 pizzas together.



In [25]:
# Test 4: Combination
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. There are 16 slices per pizza. How many slices are there?")
handle_response(response)

Calling add function
Calling multiply function
There are 112 slices.



In [31]:
# Test 5: Subtraction 
response = chat.send_message("Doug brought 4 pizzas, but Andrew dropped 2 on the ground. How many pizzas are left?")
handle_response(response)

Calling add function
There are 2 pizzas left. 2



In [35]:
# Test 6: complex substraction 
response = chat.send_message("Doug brought 10 pizzas, He gave away 3 pizzas each to his freinds Andrew and Tommy. How many pizzas are there?")
handle_response(response)

Calling add function
Calling add function
There are 4 pizzas left. 4

